<h1>Chapter 6 - Similarity Search & Vector Databases</h1>
<i>Explore different similarity search algorithms and vector databases for finding similar data.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch06_similarity_search_vector_databases/vector_databases.ipynb)

---

This notebook is for Chapter 6 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


In [ ]:
!pip install faiss-cpu==1.8.0
!pip install openai==2.14.0
!pip install chromadb==0.5.3
!pip install pandas==2.2.2
!pip install psycopg2==2.9.9

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
else:
    api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set. Add it to Colab secrets or local environment variables.")

os.environ["OPENAI_API_KEY"] = api_key

### 1 Storing and Working with Embedding using FAISS

In [ ]:
import faiss
import numpy as np
from openai import OpenAI

# Example list of sample strings
text_chunks = [
    "The sky is blue.",
    "The sun is shining.",
    "I love chocolate.",
    "Ice cream is delicious.",
    "Roses are red.",
    "Violets are blue.",
]

# Initialize the OpenAI embeddings model
client = OpenAI()
model = "text-embedding-3-small"

# Generate embeddings for the sample strings
def get_embedding(text):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

document_embeddings = np.array(
    [get_embedding(text) for text in text_chunks]
)

# Convert embeddings to float32 (FAISS requires float32 type)
document_embeddings = document_embeddings.astype("float32")

# Create a FAISS index (using L2 distance)
index = faiss.IndexFlatL2(document_embeddings.shape[1])

# Add embeddings to the index
index.add(document_embeddings)

In [ ]:
# generate a query embedding for the user query
query = "What color are violets?"
query_embedding = np.array(get_embedding(query)).astype("float32")

# Perform the search: k = number of closest documents you want to retrieve
k = 5
distances, indices = index.search(query_embedding.reshape(1, -1), k)

# Retrieve the documents corresponding to the indices
retrieved_documents = [text_chunks[i] for i in indices[0]]

In [ ]:
print("Distances: ", distances)
print("Indices: ", indices)
print("Retrieved document: ", retrieved_documents)


#### 2 Storing and Working with Embeddings in ChromaDB

In [ ]:
import chromadb

# vector store settings
VECTOR_STORE_PATH = r"../02_Data/00_Vector_Store"
COLLECTION_NAME = "my_collection"

# get/create a chroma client and collection
chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)


from openai import OpenAI

text_chunks = [
    "The sky is blue.",
    "The sun is shining.",
    "I love chocolate.",
    "Ice cream is delicious.",
    "Roses are red.",
    "Violets are blue.",
]

# Generate embeddings
client = OpenAI()
model = "text-embedding-3-small"
def get_embedding(text):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

embeddings_model = client  # for compatibility with later code
embeddings_list = [get_embedding(text) for text in text_chunks]

# add data frame to collection
collection.add(
    embeddings=embeddings_list,
    documents=text_chunks,
    ids=[
        str(i) for i in range(len(text_chunks))
    ],  # create a list of strings as index
)


# query collection
query = "What is the color of the sky?"
query_embedding = get_embedding(query)
results = collection.query(query_embeddings=[query_embedding], n_results=3)

In [ ]:
results

#### 3 Storing and Working with Embeddings in a PostgreSQL database


In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="rag_cookbook",
    user="rag_cookbook_user",
    password="rag_cookbook_user_pw",
)

cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

cur.execute(
    """
    CREATE TABLE IF NOT EXISTS embeddings (
        id SERIAL PRIMARY KEY,
        text TEXT,
        embedding VECTOR(1536)
    );
    """
)

conn.commit()
cur.close()
conn.close()

In [ ]:
conn

In [ ]:
from openai import OpenAI

# Define text chunks
text_chunks = [
    "The sky is blue.",
    "The sun is shining.",
    "I love chocolate.",
    "Ice cream is delicious.",
    "Roses are red.",
    "Violets are blue.",
]

client = OpenAI()
model = "text-embedding-3-small"

def get_embedding(text):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

index = 0
cur = conn.cursor()

# Insert the embeddings into the table
for text_chunk in text_chunks:
    embedding = get_embedding(text_chunk)
    cur.execute(
        """INSERT INTO embeddings
        (id, chunk, embedding)
        VALUES (%s, %s, %s)""",
        (index, text_chunk, embedding),
    )
    index += 1

In [ ]:
write_vectors_to_postgres(conn)

Query all data from table embeddings

In [ ]:
cur = conn.cursor()
cur.execute("SELECT * FROM embeddings_table")
rows = cur.fetchall()

for row in rows:
    print(row)

#### 4 Perform similarity search on top of the just create PostgreSQL table

In [ ]:
from openai import OpenAI

# Initialize the OpenAI embeddings model
client = OpenAI()
model = "text-embedding-3-small"

# Example text chunk
text_chunk = "Sweets are delicious."

# Get the embedding
response = client.embeddings.create(input=text_chunk, model=model)
embeded_query = response.data[0].embedding

cur = conn.cursor()
cur.execute(
    f"""
    SELECT 1 - (embedding <=> '{embeded_query}') AS cosine_similarity, *
    FROM embeddings
    ORDER BY 1 - (embedding <=> '{embeded_query}') DESC
    LIMIT 20;
"""
)

results = cur.fetchall()

In [ ]:
results

### 5 Write a larger amount of data to the vector store

For this example we are using the job description dataset to write it to the database.

In [ ]:
import psycopg2
from psycopg2 import Error

conn = psycopg2.connect(
    dbname="rag_cookbook",
    user="rag_cookbook_user",
    password="rag_cookbook_user_pw",
    host="localhost",
    port="5432",
)

cur = conn.cursor()
cur.execute("""CREATE EXTENSION IF NOT EXISTS vector""")

cur.execute(
    """
    CREATE TABLE IF NOT EXISTS job_description_table(
        id integer PRIMARY KEY,
        text_chunk TEXT,
        embedding vector(1536)
    )
    """
)
conn.commit()

In [ ]:

import psycopg2
from psycopg2 import Error
import pandas as pd

conn = psycopg2.connect(
    dbname="rag_cookbook",
    user="rag_cookbook_user",
    password="rag_cookbook_user_pw",
    host="localhost",
    port="5432",
)

cur = conn.cursor()

# execute the query to create the table including the vector column
cur.execute(
    """
    CREATE TABLE IF NOT EXISTS job_description_table(
        id integer PRIMARY KEY,
        text_chunk TEXT,
        embedding vector(1536)
    )
    """
)
conn.commit()
from openai import OpenAI

file_path = "../datasets/large_text_dataset/resume_job_description_data.csv"
df = pd.read_csv(file_path)
text_chunks = df["job_description_text"]
print(len(text_chunks))

try:
    # get all entries from the table
    cur.execute("SELECT * FROM job_description_table")
    rows = cur.fetchall()
    df_from_database = pd.DataFrame(
        rows, columns=["column1", "text_chunks", "column3"]
    )
    text_chunks_existing = df_from_database["text_chunks"]

    # compare text_chunks and text_chunks_existing and only keep the ones which are not already in the table
    text_chunks = [
        text_chunk
        for text_chunk in text_chunks
        if text_chunk not in text_chunks_existing
    ]

except:
    pass

client = OpenAI()
model = "text-embedding-3-small"

print(len(text_chunks))

# Insert the embeddings into the table using OpenAI API
for text_chunk in text_chunks:
    print(text_chunk)
    response = client.embeddings.create(input=text_chunk, model=model)
    embedding = response.data[0].embedding

    # assign a new id to the text chunk
    cur.execute("SELECT MAX(id) FROM job_description_table")
    max_id = cur.fetchone()[0]
    index = max_id + 1 if max_id else 1

    # Insert the id, content, and embedding into the table
    cur.execute(
        "INSERT INTO job_description_table (id, text_chunk, embedding) VALUES (%s, %s, %s)",
        (index, text_chunk, embedding),
    )
    conn.commit()
    print(index)

In [ ]:
results

### 6 Perform a vector search using HNSW and IVFLL index

In [ ]:
query = "I am looking for a job as a data scientist in Berlin."
client = OpenAI()
model = "text-embedding-3-small"
response = client.embeddings.create(input=query, model=model)
query_embedding = response.data[0].embedding

full_search_sql = f"""
    EXPLAIN ANALYZE SELECT 1 - (embedding <=> '{str(query_embedding)}')
    AS cosine_similarity,
    * FROM job_description_table
    ORDER BY 1 - (embedding <=> '{str(query_embedding)}') DESC
    LIMIT 20;
    """

cur.execute(full_search_sql)

results_full_search = cur.fetchall()
# [..., ('Planning Time: 13.669 ms',), ('Execution Time: 85.582 ms',)]

In [ ]:
import psycopg2
from psycopg2 import Error

ivfflat_sql = f"""
    DROP TABLE IF EXISTS test_embedding_table;
    CREATE TABLE test_embedding_table AS
        SELECT * FROM job_description_table;
    CREATE INDEX ON test_embedding_table
        USING ivfflat (embedding vector_cosine_ops)
        WITH (lists = 30);
    -- Reduce the number of probes for faster search
    SET ivfflat.probes = 3;
    EXPLAIN ANALYZE SELECT 1 - (embedding <=> '{str(query_embedding)}')
    AS cosine_similarity, *
    FROM test_embedding_table
    ORDER BY 1 - (embedding <=> '{str(query_embedding)}') DESC
    LIMIT 20;
    """

cur.execute(ivfflat_sql)
ivfflat_search = cur.fetchall()

In [ ]:
import psycopg2
from psycopg2 import Error

hnsw_test_sql = f"""
    DROP TABLE IF EXISTS test_embedding_table;
    CREATE TABLE test_embedding_table AS
        SELECT * FROM job_description_table;
    CREATE INDEX ON test_embedding_table
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 200);
    SET hnsw.ef_search = 50;
    EXPLAIN ANALYZE SELECT 1 - (embedding <=> '{str(query_embedding)}')
    AS cosine_similarity, *
    FROM test_embedding_table
    ORDER BY 1 - (embedding <=> '{str(query_embedding)}') DESC
    LIMIT 20;
"""

cur.execute(hnsw_test_sql)
hnsw_search = cur.fetchall()

# [..., ('Planning Time: 1.109 ms',), ('Execution Time: 13.927 ms',)]

In [ ]:
ivfflat_search, hnsw_search, results_full_search = perform_hnsw_similarity_search()
print(f"Index HSNW search: {hnsw_search[6]}")
print(f"Index IVFFLat search: {ivfflat_search[6]}")
print(f"Full search: {results_full_search[6]}")


### 7 Hybrid search using PostgreSQL

In [ ]:
import psycopg2
from psycopg2 import Error

conn = psycopg2.connect(
    dbname="rag_cookbook",
    user="rag_cookbook_user",
    password="rag_cookbook_user_pw",
    host="localhost",
    port="5432",
)

cur = conn.cursor()

hybrid_search_table = f"""
    DROP TABLE IF EXISTS test_embedding_table;
    CREATE TABLE test_embedding_table AS
    SELECT *, to_tsvector(text_chunk) AS tsv
    FROM job_description_table;
    """

cur.execute(hybrid_search_table)

In [ ]:
import psycopg2
from psycopg2 import Error
from openai import OpenAI

query = "I am looking for a job as a data scientist in Berlin."
client = OpenAI()
model = "text-embedding-3-small"
response = client.embeddings.create(input=query, model=model)
query_embedding = response.data[0].embedding

hybrid_search_query = f"""
    WITH ranked_docs AS (
        SELECT
            id,
            text_chunk AS text,
            ts_rank(tsv, plainto_tsquery('PostgreSQL')) AS text_score,
            1 - (embedding <=> '{str(query_embedding)}') AS vector_score,
            ts_rank(tsv, plainto_tsquery('PostgreSQL'))
            * 0.5 + (1 - (embedding <=> '{str(query_embedding)}')) * 0.5
            AS hybrid_score
        FROM test_embedding_table
        -- Filter for relevant text first
        WHERE tsv @@ plainto_tsquery('PostgreSQL')
        ORDER BY hybrid_score DESC
        LIMIT 10
    )
    SELECT * FROM ranked_docs;
    """

cur.execute(hybrid_search_query)
results = cur.fetchall()